In [1]:
import json
import os
import networkx as nx
from plan import PartialPlan
from old.run_mzn import run_mzn
from old.mzn_arr_to_schedule import *
from objects import Service, Movement
from parse_planner_output import extract_nfd_plan
from schedule_to_plan import make_new_schedule, make_new_new_schedule

In [2]:
walking_distances = {
    ("entry", "52"):3,
    ("entry", "53"):4,
    ("entry", "54"):5,
    ("entry", "55"):6,
    ("entry", "56"):7,
    ("entry", "57"):8,
    ("entry", "58"):9,
    ("entry", "59"):10,
    ("entry", "60"):8,
    ("entry", "61_service"):9,
    ("entry", "62_service"):10,
    ("entry", "63"):12,
    ("52", "53"):1,
    ("52", "54"):2,
    ("52", "55"):3,
    ("52", "56"):4,
    ("52", "57"):5,
    ("52", "58"):6,
    ("52", "59"):7,
    ("52", "60"):5,
    ("52", "61_service"):6,
    ("52", "62_service"):7,
    ("52", "63"):10,
    ("53", "54"):1,
    ("53", "55"):2,
    ("53", "56"):3,
    ("53", "57"):4,
    ("53", "58"):5,
    ("53", "59"):6,
    ("53", "60"):4,
    ("53", "61_service"):5,
    ("53", "62_service"):6,
    ("53", "63"):9,
    ("54", "55"):1,
    ("54", "56"):2,
    ("54", "57"):3,
    ("54", "58"):4,
    ("54", "59"):5,
    ("54", "60"):3,
    ("54", "61_service"):4,
    ("54", "62_service"):5,
    ("54", "63"):8,
    ("55", "56"):1,
    ("55", "57"):2,
    ("55", "58"):3,
    ("55", "59"):4,
    ("55", "60"):4,
    ("55", "61_service"):3,
    ("55", "62_service"):4,
    ("55", "63"):7,
    ("56", "57"):1,
    ("56", "58"):2,
    ("56", "59"):3,
    ("56", "60"):5,
    ("56", "61_service"):4,
    ("56", "62_service"):3,
    ("56", "63"):6,
    ("57", "58"):1,
    ("57", "59"):2,
    ("57", "60"):6,
    ("57", "61_service"):5,
    ("57", "62_service"):4,
    ("57", "63"):7,
    ("58", "59"):1,
    ("58", "60"):7,
    ("58", "61_service"):6,
    ("58", "62_service"):5,
    ("58", "63"):8,
    ("59", "60"):8,
    ("59", "61_service"):7,
    ("59", "62_service"):6,
    ("59", "63"):9,
    ("60", "61_service"):1,
    ("60", "62_service"):2,
    ("60", "63"):4,
    ("61_service", "62_service"):1,
    ("61_service", "63"):3,
    ("62_service", "63"):4,
}

In [3]:
rows = list()
for cfg in range(1, 50):
    print(f'\n\n\nConfig {cfg}\n===============\n\n\n')
    for num_t in range(3, 16):

        plan_file = f"../results/nfd/base3/pln_{cfg}_{num_t}t.txt"
        
        if not os.path.exists(plan_file):
            continue
        
        plan = extract_nfd_plan(plan_file)


        solved = len(plan) > 0
        if solved:

            pp = PartialPlan(plan)
            pp.build_constraints()
            pp.build_walking_times_matrix(walking_distances)
            pp.write_dzn(1)

            if len(pp.actions) < 1:
                continue

            [start_times, durations, action_driver, action_train] = run_mzn(300, 'chuffed')
            train_schedule, driver_schedule = init_train_driver_schedules(start_times,durations, 
                                                                            action_train, action_driver)
            
            pp_dur = max([start_times[i]+durations[i] for i in range(len(start_times))])


            new_schedule = make_new_schedule(pp, start_times)
            new_new_schedule = make_new_new_schedule(new_schedule)
            with open(f'../results/nfd/base3/plans/plan_{cfg}_{num_t}t', 'w') as f:
                f.writelines([str(l)+'\n' for l in new_new_schedule])
            plot_schedule2(train_schedule, pp.actions, f'../results/nfd/base3/plots/plot_{cfg}_{num_t}t')

            row = {'config':cfg,'num_trains':num_t,'makespan_pp':int(pp_dur)}
                

            rows.append(row)

df = pd.DataFrame(rows)
if not os.path.exists('results_base3_nfd.csv'):
    df.to_csv('results_base3_nfd.csv', index=False)
else:
    df_old = pd.read_csv('results_base3_nfd.csv')
    df_new = pd.concat([df,df_old], ignore_index=True)
    df_new = df_new.drop_duplicates(subset=['config','num_trains'])
    df_new.to_csv('results_base3_nfd.csv', index=False)




Config 1



{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 54, "flatIntVars": 12, "flatBoolConstraints": 27, "flatIntConstraints": 102, "evaluatedReifiedConstraints": 54, "method": "minimize", "flatTime": 0.24726}}
{"type": "statistics", "statistics": {"nSolutions": 1}}


{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 108, "flatIntVars": 16, "flatBoolConstraints": 54, "flatIntConstraints": 172, "evaluatedReifiedConstraints": 108, "method": "minimize", "flatTime": 0.119393}}
{"type": "statistics", "statistics": {"nSolutions": 1}}


{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 180, "flatIntVars": 20, "flatBoolConstraints": 90, "flatIntConstraints": 262, "evaluatedReifiedConstraints": 180, "method": "minimize", "flatTime": 0.111734}}
{"type": "statistics", "statistics": {"nSolutions": 1}}


{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 270, "flatIntVars": 24, "flatBoolConstraints": 135, "flatIntConstraints": 382, "evaluatedReifiedConstraints": 270, "method": "minimize", "flatTime": 0.124885}}
{"type": "statistics", "statistics": {"nSolutions": 1}}


{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 378, "flatIntVars": 28, "flatBoolConstraints": 189, "flatIntConstraints": 514, "evaluatedReifiedConstraints": 378, "method": "minimize", "flatTime": 0.156505}}
{"type": "statistics", "statistics": {"nSolutions": 1}}


...




Config 16



{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 54, "flatIntVars": 12, "flatBoolConstraints": 27, "flatIntConstraints": 101, "evaluatedReifiedConstraints": 54, "method": "minimize", "flatTime": 0.113726}}
{"type": "statistics", "statistics": {"nSolutions": 1}}


{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 108, "flatIntVars": 16, "flatBoolConstraints": 54, "flatIntConstraints": 175, "evaluatedReifiedConstraints": 108, "method": "minimize", "flatTime": 0.118872}}
{"type": "statistics", "statistics": {"nSolutions": 1}}


{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 180, "flatIntVars": 20, "flatBoolConstraints": 90, "flatIntConstraints": 271, "evaluatedReifiedConstraints": 180, "method": "minimize", "flatTime": 0.144413}}
{"type": "statistics", "statistics": {"nSolutions": 1}}


{"type": "statistics", "statistics": {"paths": 0, "flatBoolVars": 270, "flatIntVars": 24, "flatBoolConstraints": 135, "flatIntConstraints": 376, "evaluatedReifiedConstraints": 270, "method": "minimize", "flatTime": 0.122637}}
{"type": "statistics", "statistics": {"nSolutions": 2}}





Config 17






Config 18






Config 19






Config 20






Config 21






Config 22






Config 23






Config 24






Config 25






Config 26






Config 27






Config 28






Config 29






Config 30






Config 31






Config 32






Config 33






Config 34






Config 35






Config 36






Config 37






Config 38






Config 39






Config 40






Config 41






Config 42






Config 43






Config 44






Config 45






Config 46






Config 47






Config 48






Config 49



